<a href="https://colab.research.google.com/github/PzykoEich/proyecto_NLP/blob/main/Untitled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# En este momento no se que debo hacer 😅

# Verificación de emociones

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import nltk
import spacy
import kagglehub
import random
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re

2026-03-19 15:31:32.642303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773955892.668068     988 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773955892.678011     988 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773955892.766451     988 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773955892.766482     988 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773955892.766485     988 computation_placer.cc:177] computation placer alr

In [2]:
#Load data from kaggle
path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis", path='twitter_training.csv')

In [3]:
df = pd.read_csv(path, header=None)
df.columns = ['id','juego','sentimiento','comentario']

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           74682 non-null  int64
 1   juego        74682 non-null  str  
 2   sentimiento  74682 non-null  str  
 3   comentario   73996 non-null  str  
dtypes: int64(1), str(3)
memory usage: 2.3 MB


In [5]:
df

,id,juego,sentimiento,comentario
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
...,...,...,...,...
74677,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74678,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74679,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74680,9200,Nvidia,Positive,Just realized between the windows partition of...


In [6]:
df['sentimiento'].value_counts()

sentimiento
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [7]:
df['comentario'] = df['comentario'].astype(str)

In [8]:
#Estableciendo semillas

seed = 40
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [9]:
nltk.download('punkt')  #tokenizer de nltk
nltk.download('punkt_tab')  #tokenizer de nltk
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/martin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/martin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/martin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
stopwords_en = stopwords.words('english')

In [11]:
#Limpieza de texto
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"http\S+", "", texto)
    texto = re.sub(r"@\w+", "", texto)
    texto = re.sub(r"#\w+", "", texto)
    texto = re.sub(r"[^a-záéíóúñ ]", "", texto)
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stopwords_en]
    return " ".join(palabras)

In [12]:
df['comentario'] = df['comentario'].astype(str)

In [13]:
df['comentario'] = df['comentario'].fillna("")

In [14]:
df['comentario_limpio'] = df['comentario'].apply(limpiar_texto)

In [15]:
df

,id,juego,sentimiento,comentario,comentario_limpio
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,im getting borderlands murder
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,coming borders kill
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,im getting borderlands kill
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,im coming borderlands murder
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,im getting borderlands murder
...,...,...,...,...,...
74677,9200,Nvidia,Positive,Just realized that the Windows partition of my...,realized windows partition mac like years behi...
74678,9200,Nvidia,Positive,Just realized that my Mac window partition is ...,realized mac window partition years behind nvi...
74679,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...,realized windows partition mac years behind nv...
74680,9200,Nvidia,Positive,Just realized between the windows partition of...,realized windows partition mac like years behi...


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["comentario_limpio"])
y = df["sentimiento"]


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

modelo = LogisticRegression(max_iter=200)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)
print(classification_report(y_test, pred))


              precision    recall  f1-score   support

  Irrelevant       0.68      0.52      0.59      2598
    Negative       0.73      0.76      0.75      4509
     Neutral       0.61      0.66      0.64      3664
    Positive       0.69      0.71      0.70      4166

    accuracy                           0.68     14937
   macro avg       0.68      0.66      0.67     14937
weighted avg       0.68      0.68      0.68     14937



In [18]:
df = pd.get_dummies(df, columns=["juego"])

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(128, activation='relu', input_shape=(X.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

/home/martin/miniconda3/envs/ciencia/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1773955905.167092     988 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [20]:
df['sentimiento'].unique()

<StringArray>
['Positive', 'Neutral', 'Negative', 'Irrelevant']
Length: 4, dtype: str

In [21]:
df["sentimiento"] = df["sentimiento"].map({"Positive": 1, "Negative": 0, "Neutral": 2, "Irrelevant": 3 })
y = df["sentimiento"].values


In [22]:
X = vectorizer.fit_transform(df['comentario_limpio']).toarray()
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [23]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

2026-03-19 15:31:55.464048: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 955920000 exceeds 10% of free system memory.
2026-03-19 15:31:57.687703: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 955920000 exceeds 10% of free system memory.


Epoch 1/10


I0000 00:00:1773955920.973129    1050 service.cc:152] XLA service 0x747d18004a60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773955920.976495    1050 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-03-19 15:32:01.119914: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773955921.536918    1050 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-19 15:32:02.420301: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_127', 12 bytes spill stores, 12 bytes spill loads

2026-03-19 15:32:02.618284: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion

  37/1494 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.3007 - loss: 0.5919

I0000 00:00:1773955923.698196    1050 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1477/1494 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2783 - loss: -1518.4868

2026-03-19 15:32:08.653965: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_127', 12 bytes spill stores, 12 bytes spill loads

2026-03-19 15:32:08.894096: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_127', 4 bytes spill stores, 4 bytes spill loads



1494/1494 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.2778 - loss: -6038.3389 - val_accuracy: 0.2788 - val_loss: -23935.9043
Epoch 2/10
1494/1494 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.2777 - loss: -75046.1641 - val_accuracy: 0.2788 - val_loss: -150090.5469
Epoch 3/10
1494/1494 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.2777 - loss: -265723.5938 - val_accuracy: 0.2788 - val_loss: -414712.0312
Epoch 4/10
1494/1494 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.2777 - loss: -604593.0625 - val_accuracy: 0.2788 - val_loss: -840145.5000
Epoch 5/10
1494/1494 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.2777 - loss: -1113636.2500 - val_accuracy: 0.2788 - val_loss: -1449290.2500
Epoch 6/10
1494/1494 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.2777 - loss: -1816002.8750 - val_accuracy: 0.2788 - val_loss: -2265464.7500
Epoch 7/10
1494/1494 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.2777 - loss: -2734710.0000 - val_accuracy: 0.2788 - val_loss: -3312085.0000
Epoch 8/10
149

In [24]:
print(df["sentimiento"].unique())


[1 2 0 3]
